In [3]:
import mujoco
model = mujoco.MjModel.from_xml_path(
    "mujoco_playground/_src/locomotion/pendulum_inverse_pendulum/xmls/pendulum_inverse_pendulum.xml"
)
print(model.nq, model.nv, model.nu)  # should print your joint/actuator counts

2 2 2


In [4]:
import time

import jax
import jax.numpy as jp
from mujoco_playground._src.locomotion.pendulum_inverse_pendulum.environment import DoublePendulumEnv

# Check JAX devices
print("JAX devices:", jax.devices())

# 1. Create the environment
env = DoublePendulumEnv()
print("✓ Environment created")
print(f"  action_size: {env.action_size}")
print(f"  xml_path: {env.xml_path}")
print(f"  n_substeps: {env.n_substeps}")

# 2. Test reset
rng = jax.random.PRNGKey(0)
state = env.reset(rng)
print("\n✓ Reset successful")
print(f"  obs shape:    {state.obs.shape}")       # should be (6,)
print(f"  reward:       {state.reward}")          # should be 0.0
print(f"  done:         {state.done}")            # should be 0.0

# 3. Test a single step with random action
rng, action_rng = jax.random.split(rng)
action = jax.random.uniform(action_rng, (env.action_size,), minval=-1.0, maxval=1.0)
state = env.step(state, action)
print("\n✓ Step successful")
print(f"  obs shape:    {state.obs.shape}")       # should be (6,)
print(f"  reward:       {state.reward}")          # should be a float
print(f"  done:         {state.done}")            # should be 0.0 unless NaN

# 4. Test a full rollout (10 steps to see if JIT helps)
print("\nRunning 10 steps...")
rng, action_rng = jax.random.split(rng)
action = jax.random.uniform(action_rng, (env.action_size,), minval=-1.0, maxval=1.0)
for i in range(10):
    step_time_start = time.time()
    state = env.step(state, action)
    step_time_end = time.time()
    print(f"Step {i+1:2d}: reward={state.reward:.4f}, done={state.done}, time={step_time_end - step_time_start:.4f}")

print(f"✓ 10 steps completed, final reward: {state.reward:.4f}")

JAX devices: [CpuDevice(id=0)]


AttributeError: property 'n_substeps' of 'DoublePendulumEnv' object has no setter